In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class AliphaticEpoxidation(MorphingOperator):
    def __init__(self):
        super(AliphaticEpoxidation, self).__init__()
        self._name = "Aliphatic Epoxidation (Phase I - Stereospecific)"
        self._target_bonds = []
        self.DOUBLE_BOND_PATTERN = Chem.MolFromSmarts("[CX3;!a]=[CX3;!a]")

    def setOriginal(self, mol):
        super(AliphaticEpoxidation, self).setOriginal(mol)
        self._target_bonds = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        if self.DOUBLE_BOND_PATTERN is not None:
            matches = rdkit_mol.GetSubstructMatches(self.DOUBLE_BOND_PATTERN)
            for match in matches:
                
                pair = tuple(sorted([match[0], match[1]]))
                if pair not in self._target_bonds:
                    self._target_bonds.append(pair)

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._target_bonds:
            return MolpherMol(other=rdkit_mol)
        
        idx1, idx2 = random.choice(self._target_bonds)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx1, idx2)
            if bond:
                bond.SetBondType(Chem.BondType.SINGLE)
            
            oxygen_idx = rw_mol.AddAtom(Chem.Atom(8))
            
            rw_mol.AddBond(idx1, oxygen_idx, Chem.BondType.SINGLE)
            rw_mol.AddBond(idx2, oxygen_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()

            for idx in [idx1, idx2, oxygen_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
    
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name

epox_op = AliphaticEpoxidation()

test_epoxidation_molecules = {
    "1. 2-Βουτένιο (Αλειφατικό Αλκένιο -> Epoxide)": "CC=CC",
    "2. Κυκλοεξένιο (Διπλός δεσμός σε δακτύλιο -> Εποξείδωση χωρίς διάσπαση)": "C1=CCCCC1",
    "3. Βενζόλιο (Αρωματικό -> Πρέπει να αγνοηθεί πλήρως)": "C1=CC=CC=C1"
}

print("=== STARTING ALIPHATIC EPOXIDATION TESTING ===")
for name, smiles in test_epoxidation_molecules.items():
    mol = MolpherMol(smiles)
    epox_op.setOriginal(mol)
    product = epox_op.morph()
    
    print(f"\n{name}")
    print(f"  SOURCE: {mol.getSMILES()}")
    print(f"  TARGET: {product.getSMILES() if product and product.getSMILES() != mol.getSMILES() else 'No change (Safe)'}")
print("\n==============================================")

=== STARTING ALIPHATIC EPOXIDATION TESTING ===

1. 2-Βουτένιο (Αλειφατικό Αλκένιο -> Epoxide)
  SOURCE: CC=CC
  TARGET: CC1OC1C

2. Κυκλοεξένιο (Διπλός δεσμός σε δακτύλιο -> Εποξείδωση χωρίς διάσπαση)
  SOURCE: C1=CCCCC1
  TARGET: C1CCC2OC2C1

3. Βενζόλιο (Αρωματικό -> Πρέπει να αγνοηθεί πλήρως)
  SOURCE: C1=CC=CC=C1
  TARGET: No change (Safe)

